In [9]:
import pandas as pd
from tqdm import tqdm

import csv
import re
from pathlib import Path

In [10]:
RAW_ROOT = Path("/home/aniket/Programming/renters/data/cmhc-rental-raw")
CLEAN_ROOT = Path("/home/aniket/Programming/renters/data/cmhc-rental-clean")

SUBFOLDERS = ["avg-rent", "pct-change-rent", "vacancy-rate"]
COLUMNS = ["Year", "Studio", "1 Bedroom", "2 Bedroom", "3 Bedroom+", "Total"]

In [11]:
year_re = re.compile(r"^(\d{4})\s+October\b", re.IGNORECASE)

def _parse_value(raw: str, is_pct: bool):
    if raw is None:
        return None
    value = raw.strip()
    if not value or value == "**":
        return None
    if value == "++":
        return 0.0
    value = value.replace(",", "")
    try:
        num = float(value)
    except ValueError:
        return None
    if is_pct:
        return float(num)
    if num.is_integer():
        return int(num)
    return int(round(num))

def _row_value(row, index):
    if index < len(row):
        return row[index]
    return ""

def parse_cmhc_csv(file_path: Path, is_pct: bool) -> pd.DataFrame:
    with file_path.open("r", encoding="utf-8", errors="replace", newline="") as handle:
        rows = list(csv.reader(handle))

    year_map = {}
    max_year = None
    for row in rows:
        if not row:
            continue
        first_cell = row[0].strip() if len(row) > 0 else ""
        match = year_re.match(first_cell)
        if not match:
            continue
        year = int(match.group(1))
        studio = _parse_value(_row_value(row, 1), is_pct)
        one_bed = _parse_value(_row_value(row, 3), is_pct)
        two_bed = _parse_value(_row_value(row, 5), is_pct)
        three_bed = _parse_value(_row_value(row, 7), is_pct)
        total = _parse_value(_row_value(row, 9), is_pct)
        year_map[year] = [year, studio, one_bed, two_bed, three_bed, total]
        max_year = year if max_year is None else max(max_year, year)

    if max_year is None:
        return pd.DataFrame(columns=COLUMNS)

    data_rows = []
    for year in range(1990, max_year + 1):
        data_rows.append(year_map.get(year, [year, None, None, None, None, None]))

    return pd.DataFrame(data_rows, columns=COLUMNS)

def clean_all_files():
    for subfolder in SUBFOLDERS:
        source_dir = RAW_ROOT / subfolder
        target_dir = CLEAN_ROOT / subfolder
        target_dir.mkdir(parents=True, exist_ok=True)

        files = sorted(source_dir.glob("*.csv"))
        is_pct = subfolder in ("pct-change-rent", "vacancy-rate")
        for file_path in tqdm(files, desc=f"Cleaning {subfolder}"):
            df = parse_cmhc_csv(file_path, is_pct)
            out_path = target_dir / file_path.name
            if is_pct:
                for col in COLUMNS[1:]:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
                df.to_csv(out_path, index=False, na_rep="", float_format="%.1f")
            else:
                for col in COLUMNS[1:]:
                    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
                df.to_csv(out_path, index=False, na_rep="")

clean_all_files()

Cleaning vacancy-rate: 100%|██████████| 34/34 [00:00<00:00, 548.38it/s]


In [12]:
summary_dir = CLEAN_ROOT / "summary"
summary_dir.mkdir(parents=True, exist_ok=True)

avg_dir = CLEAN_ROOT / "avg-rent"
pct_dir = CLEAN_ROOT / "pct-change-rent"
vac_dir = CLEAN_ROOT / "vacancy-rate"

def _first_year_with_value(df, column="Total"):
    mask = df[column].notna()
    if not mask.any():
        return None
    return int(df.loc[mask, "Year"].iloc[0])

def _pct_increase(start, end):
    if pd.isna(start) or pd.isna(end) or start == 0:
        return None
    return (end - start) / start * 100.0

def _value_for_year(df, year):
    match = df.loc[df["Year"] == year, "Total"]
    return match.iloc[0] if len(match) else None

summary_rows = []
avg_files = sorted(avg_dir.glob("*.csv"))

for avg_path in tqdm(avg_files, desc="Summarizing cities"):
    city = avg_path.stem
    pct_path = pct_dir / avg_path.name
    vac_path = vac_dir / avg_path.name

    avg_df = pd.read_csv(avg_path)
    pct_df = pd.read_csv(pct_path) if pct_path.exists() else pd.DataFrame(columns=avg_df.columns)
    vac_df = pd.read_csv(vac_path) if vac_path.exists() else pd.DataFrame(columns=avg_df.columns)

    first_year_avg = _first_year_with_value(avg_df, "Total")
    first_year_pct = _first_year_with_value(pct_df, "Total") if not pct_df.empty else None
    first_year_vac = _first_year_with_value(vac_df, "Total") if not vac_df.empty else None

    avg_total = pd.to_numeric(avg_df.get("Total"), errors="coerce")
    pct_total = pd.to_numeric(pct_df.get("Total"), errors="coerce") if not pct_df.empty else pd.Series(dtype="float64")
    vac_total = pd.to_numeric(vac_df.get("Total"), errors="coerce") if not vac_df.empty else pd.Series(dtype="float64")

    avg_stats = {
        "avg_total_count": int(avg_total.count()),
        "avg_total_mean": avg_total.mean(),
        "avg_total_median": avg_total.median(),
        "avg_total_min": avg_total.min(),
        "avg_total_max": avg_total.max(),
        "avg_total_std": avg_total.std(),
        "avg_total_2025": avg_df.loc[avg_df["Year"] == 2025, "Total"].squeeze() if 2025 in avg_df["Year"].values else None,
    }

    pct_stats = {
        "pct_total_count": int(pct_total.count()),
        "pct_total_mean": pct_total.mean(),
        "pct_total_median": pct_total.median(),
        "pct_total_min": pct_total.min(),
        "pct_total_max": pct_total.max(),
        "pct_total_std": pct_total.std(),
    }

    vac_stats = {
        "vac_total_count": int(vac_total.count()),
        "vac_total_mean": vac_total.mean(),
        "vac_total_median": vac_total.median(),
        "vac_total_min": vac_total.min(),
        "vac_total_max": vac_total.max(),
        "vac_total_std": vac_total.std(),
    }

    inc_2004_2011 = _pct_increase(_value_for_year(avg_df, 2004), _value_for_year(avg_df, 2011))
    inc_2011_2018 = _pct_increase(_value_for_year(avg_df, 2011), _value_for_year(avg_df, 2018))
    inc_2004_2025 = _pct_increase(_value_for_year(avg_df, 2004), _value_for_year(avg_df, 2025))
    inc_2018_2025 = _pct_increase(_value_for_year(avg_df, 2018), _value_for_year(avg_df, 2025))

    summary_rows.append({
        "City": city,
        "first_year_avg_total": first_year_avg,
        "first_year_pct_total": first_year_pct,
        "first_year_vac_total": first_year_vac,
        "pct_increase_total_2004_2011": inc_2004_2011,
        "pct_increase_total_2011_2018": inc_2011_2018,
        "pct_increase_total_2018_2025": inc_2018_2025,
        "pct_increase_total_2004_2025": inc_2004_2025,
        **avg_stats,
        **pct_stats,
        **vac_stats,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("City").reset_index(drop=True)
summary_df = summary_df.round(2)
summary_out = summary_dir / "city_summary.csv"
summary_df.to_csv(summary_out, index=False)

summary_df.head()

Summarizing cities:  44%|████▍     | 15/34 [00:00<00:00, 146.50it/s]/home/aniket/miniconda3/envs/py312/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/aniket/miniconda3/envs/py312/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/aniket/miniconda3/envs/py312/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
Summarizing cities: 100%|██████████| 34/34 [00:00<00:00, 152.92it/s]


,City,first_year_avg_total,first_year_pct_total,first_year_vac_total,pct_increase_total_2004_2011,pct_increase_total_2011_2018,pct_increase_total_2018_2025,pct_increase_total_2004_2025,avg_total_count,avg_total_mean,...,pct_total_median,pct_total_min,pct_total_max,pct_total_std,vac_total_count,vac_total_mean,vac_total_median,vac_total_min,vac_total_max,vac_total_std
0,ajax,1990.0,1991.0,1990.0,1.17,20.69,38.76,69.43,36,1047.89,...,2.9,-1.9,10.5,2.84,36,1.86,1.60,0.0,5.9,1.40
1,barrie,1990.0,1991.0,1990.0,9.94,32.37,31.91,91.98,36,996.75,...,3.2,0.0,6.7,1.74,36,2.44,2.20,0.6,4.4,1.01
2,brampton,1990.0,1991.0,1990.0,7.09,22.31,49.54,95.85,36,1103.61,...,2.4,-0.7,6.3,1.94,36,2.07,1.75,0.4,4.3,1.10
3,brantford,1990.0,1991.0,1990.0,14.80,25.91,53.98,122.56,34,832.82,...,3.2,-1.3,10.8,2.51,34,2.71,2.40,0.6,7.0,1.21
4,burlington,1990.0,1991.0,1990.0,15.99,28.12,41.82,110.77,36,1092.28,...,3.6,0.0,7.1,1.82,36,1.36,1.35,0.2,2.4,0.54
